# S5Mars — QAT INT8/FP16 selettivo con NVIDIA ModelOpt

Notebook minimale per **SegFormer-B0**:

1. carica il checkpoint FP32 e i dati senza oversampling/augmentation;
2. quantizza i `Conv2d`, gli MLP dell'encoder e le proiezioni del decoder, lasciando attenzione, LayerNorm, Softmax, Add e Resize in FP16;
3. esegue QAT con loss supervisionata + distillazione;
4. esporta il modello QAT in ONNX Q/DQ e materializza i pesi costanti INT8;
5. costruisce l'engine TensorRT INT8 Q/DQ e, solo se richiesto, la baseline FP16;
6. confronta mIoU, latenza, energia, immagini/J, memoria e dimensione engine.

Non contiene PTQ o `quantize_static`; l'unico post-process ONNX è il folding equivalente dei pesi costanti `Q → DQ` in initializer `INT8 → DQ`. Impostando `RESTORE_SAVED_QAT=True` il confronto può essere ripetuto dal best checkpoint senza nuovo training.

## 1. Dipendenze

Le versioni di ModelOpt, ONNX Runtime GPU e TensorRT sono fissate perché exporter, Q/DQ e API del builder devono essere compatibili con CUDA 12. Dopo l'installazione su Kaggle riavviare la sessione prima di proseguire.

In [ ]:
# Una sola implementazione di ONNX Runtime evita conflitti tra pacchetti CPU e GPU.
%pip uninstall -y -q onnxruntime onnxruntime-gpu
# Serve modelopt.torch: l'extra [onnx] non è necessario e imporrebbe una diversa versione di ORT.
%pip install -U -q "nvidia-modelopt==0.46.1" onnx==1.21.0 onnxruntime-gpu==1.26.0 tensorrt-cu12==10.9.0.34 nvidia-ml-py datasets transformers google-api-python-client google-auth tqdm

## 2. Configurazione

Questa è l'unica cella da modificare tra esperimenti. Seleziona checkpoint, gruppi Linear INT8, fallback FP16, training, gate di parità e benchmark. `RUN_FINAL_TEST` resta False finché variante ed epoca non sono state scelte esclusivamente sulla validation.

In [ ]:
REPO_ID = "Mirali33/mb-s5mars"
TRAIN_SPLIT, VAL_SPLIT, TEST_SPLIT = "train", "val", "test"
IMAGE_SIZE = (512, 512)
IMAGE_MEAN = (0.485, 0.456, 0.406)
IMAGE_STD = (0.229, 0.224, 0.225)
NUM_CLASSES = 9
IGNORE_INDEX = -100  # Non compare nelle mask 0..8: tutte le classi entrano nelle metriche.
CLASS_NAMES = {
    0: "Background", 1: "Bedrock", 2: "Hole", 3: "Ridge",
    4: "Rock", 5: "Rover", 6: "Sand / Soil", 7: "Sky", 8: "Track",
}

MODEL_NAME = "segformer_b0"
PRETRAINED_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"
EXPERIMENT = MODEL_NAME
QAT_VARIANT = "int8_conv_mlp_decoder_linear_qdq"
# Secondo esperimento SegFormer: Conv e Linear a maggiore costo, attenzione ancora FP16.
INT8_LINEAR_GROUPS = ("encoder_mlp", "decoder_projection")
FP16_FALLBACK_MODULE_PREFIXES = ()
CHECKPOINT_FILENAME = "best.ckpt"
CHECKPOINT_FOLDER_ID = "1gor4_Lct_xBMJuWlfxnAXm0fYNjcKBRs"

SEED = 42
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 1
CALIBRATION_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
NUM_WORKERS = 2
CALIBRATION_SAMPLES = 500
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
WARMUP_EPOCHS = 1
USE_AMP = True

# Loss già concordata: 70% supervisione e 30% distillazione dal modello FP32.
SUPERVISED_WEIGHT = 0.7
DISTILLATION_WEIGHT = 0.3
DISTILLATION_TEMPERATURE = 2.0
COMBINED_LOSS_ALPHA = 0.5
DICE_SMOOTH = 1e-5

MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None
RESTORE_SAVED_QAT = False  # True: salta calibrazione/training e riparte dal best ModelOpt su Drive.
RUN_TRT_FP16_BASELINE = False  # Opzionale: il confronto corrente usa i risultati FP16 già salvati.
RUN_FINAL_TEST = False  # Attivare solo dopo la scelta finale sulla validation.

# Baseline QAT Conv-only già misurata: serve per decidere il nuovo esperimento sulla validation.
SEGFORMER_QAT_CONV_ONLY_REFERENCE = {
    "validation_miou": 0.8300365905100314,
    "mean_latency_ms": 5.65719264004656,
    "p95_latency_ms": 7.511129749673273,
    "average_sampled_power_w": 67.1286606997559,
    "energy_per_image_mj": 291.7556744253289,
    "images_per_joule": 3.4275254524858854,
    "peak_gpu_memory_mib": 1008.0,
    "engine_size_mib": 10.90683364868164,
    "test_miou": 0.7692888447846535,
}
MAX_VALIDATION_MIOU_DROP_VS_CONV_ONLY = 0.005

OUTPUT_DIR = f"/kaggle/working/qat_modelopt/{EXPERIMENT}/{QAT_VARIANT}"
CHECKPOINT_DIR = f"{OUTPUT_DIR}/checkpoints"
ONNX_PATH = f"{OUTPUT_DIR}/{EXPERIMENT}_{QAT_VARIANT}.onnx"
BEST_QAT_PATH = f"{CHECKPOINT_DIR}/best_modelopt_qat.pth"
LAST_QAT_PATH = f"{CHECKPOINT_DIR}/last_modelopt_qat.pth"
HISTORY_PATH = f"{OUTPUT_DIR}/history.json"
RESULTS_PATH = f"{OUTPUT_DIR}/results.json"
ENGINE_PATH = f"{OUTPUT_DIR}/{EXPERIMENT}_{QAT_VARIANT}.engine"
FP16_ONNX_PATH = f"{OUTPUT_DIR}/{EXPERIMENT}_{QAT_VARIANT}_qat_weights_fp16.onnx"
FP16_ENGINE_PATH = f"{OUTPUT_DIR}/{EXPERIMENT}_{QAT_VARIANT}_qat_weights_fp16.engine"

QAT_DRIVE_FOLDER_NAME = f"s5mars_qat_modelopt_{EXPERIMENT}_{QAT_VARIANT}"
QAT_DRIVE_PARENT_FOLDER_ID = None
UPLOAD_BEST_EACH_EPOCH = True
UPLOAD_FINAL_ARTIFACTS = True

ONNX_OPSET = 20  # Versione usata dall'esempio vision ufficiale ModelOpt.
FP16_FALLBACK_MIN_PREDICTION_AGREEMENT = 0.95
EXPORT_MIN_PREDICTION_AGREEMENT = 0.95
TRT_MIN_PREDICTION_AGREEMENT = 0.95
TRT_DEVICE_ID = 0
TRT_WORKSPACE_GIB = 4
# Workaround compatibile con TensorRT 10.9; il weak typing non sarà disponibile in TensorRT 11.
TRT_BUILD_MODE = "weakly_typed_explicit_qdq"
TRT_FP16_BUILD_MODE = "weakly_typed_fp16"
TRT_BUILDER_OPTIMIZATION_LEVEL = 5  # Esplora il numero massimo di tattiche disponibili.
LATENCY_WARMUP_RUNS = 10
LATENCY_MEASURED_RUNS = 50
EFFICIENCY_WARMUP_RUNS = 10
EFFICIENCY_MIN_RUNS = 100
EFFICIENCY_MIN_SECONDS = 30.0
NVML_SAMPLE_INTERVAL_SECONDS = 0.02

assert abs(SUPERVISED_WEIGHT + DISTILLATION_WEIGHT - 1.0) < 1e-12

## 3. Runtime

Import, seed e controlli bloccanti del runtime. Il notebook richiede CUDA e ONNX Runtime GPU; versioni e provider effettivi vengono stampati per rendere il run riproducibile.

In [ ]:
import copy
import gc
import hashlib
import importlib.metadata
import io
import json
import math
import os
import random
import threading
import time
from pathlib import Path

import numpy as np
import onnx
from onnx import numpy_helper
import tensorrt as trt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build as build_google_api
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

import modelopt.torch.opt as mto
import modelopt.torch.quantization as mtq
from modelopt.torch.quantization.nn import TensorQuantizer
from transformers import SegformerConfig, SegformerForSemanticSegmentation
import onnxruntime as ort
import pynvml


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
DEVICE = torch.device("cuda")
if not torch.cuda.is_available():
    raise RuntimeError("Il QAT e il benchmark richiedono una GPU CUDA")
required_providers = {"CUDAExecutionProvider"}
missing_providers = required_providers - set(ort.get_available_providers())
if missing_providers:
    raise RuntimeError(f"Execution Provider mancanti: {sorted(missing_providers)}")

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("modelopt", importlib.metadata.version("nvidia-modelopt"))
print("transformers", importlib.metadata.version("transformers"))
print("onnx", onnx.__version__, "onnxruntime", ort.__version__, "tensorrt", trt.__version__)
print("providers", ort.get_available_providers())

## 4. Checkpoint e persistenza su Google Drive

Le credenziali OAuth e il token Hugging Face provengono soltanto da variabili d'ambiente o Kaggle Secrets. Il checkpoint FP32 sorgente è separato dagli artefatti QAT; `RESTORE_SAVED_QAT=True` ripristina pesi e stato dei quantizer senza riaddestrare.

In [ ]:
def get_secret(name, env_name=None, required=True):
    value = os.environ.get(env_name or name)
    if not value:
        try:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(name)
        except Exception:
            value = None
    if required and not value:
        raise RuntimeError(f"Secret non disponibile: {name}")
    return value


def drive_service():
    credentials = Credentials(
        token=None,
        refresh_token=get_secret("GDRIVE_REFRESH_TOKEN"),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=get_secret("GDRIVE_CLIENT_ID"),
        client_secret=get_secret("GDRIVE_CLIENT_SECRET"),
    )
    return build_google_api("drive", "v3", credentials=credentials, cache_discovery=False)


def drive_query_escape(value):
    # Drive query literals escape both backslashes and single quotes.
    return str(value).replace("\\", "\\\\").replace("'", "\\'")


def download_checkpoint():
    destination = Path(CHECKPOINT_DIR) / f"source_{CHECKPOINT_FILENAME}"
    if destination.is_file():
        return destination
    service = drive_service()
    response = service.files().list(
        q=(
            f"'{drive_query_escape(CHECKPOINT_FOLDER_ID)}' in parents and "
            f"name = '{drive_query_escape(CHECKPOINT_FILENAME)}' and trashed = false"
        ),
        fields="files(id,name)", pageSize=10,
    ).execute()
    files = response.get("files", [])
    if len(files) != 1:
        raise FileNotFoundError(f"Atteso un solo {CHECKPOINT_FILENAME}, trovati {len(files)}")
    request = service.files().get_media(fileId=files[0]["id"])
    with io.FileIO(destination, "wb") as stream:
        downloader = MediaIoBaseDownload(stream, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return destination


def resolve_output_folder(service):
    escaped = drive_query_escape(QAT_DRIVE_FOLDER_NAME)
    clauses = [
        f"name = '{escaped}'",
        "mimeType = 'application/vnd.google-apps.folder'",
        "trashed = false",
    ]
    if QAT_DRIVE_PARENT_FOLDER_ID:
        clauses.append(f"'{drive_query_escape(QAT_DRIVE_PARENT_FOLDER_ID)}' in parents")
    files = service.files().list(
        q=" and ".join(clauses), fields="files(id,name)", pageSize=10,
    ).execute().get("files", [])
    if len(files) > 1:
        raise RuntimeError(f"Più cartelle Drive chiamate {QAT_DRIVE_FOLDER_NAME}")
    if files:
        return files[0]["id"]
    metadata = {
        "name": QAT_DRIVE_FOLDER_NAME,
        "mimeType": "application/vnd.google-apps.folder",
    }
    if QAT_DRIVE_PARENT_FOLDER_ID:
        metadata["parents"] = [QAT_DRIVE_PARENT_FOLDER_ID]
    return service.files().create(body=metadata, fields="id").execute()["id"]


def upload_file(path, service=None, folder_id=None):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    service = service or drive_service()
    folder_id = folder_id or resolve_output_folder(service)
    escaped = drive_query_escape(path.name)
    escaped_folder_id = drive_query_escape(folder_id)
    files = service.files().list(
        q=f"'{escaped_folder_id}' in parents and name = '{escaped}' and trashed = false",
        fields="files(id,name)", pageSize=10,
    ).execute().get("files", [])
    if len(files) > 1:
        raise RuntimeError(f"Più file Drive chiamati {path.name!r} nella stessa cartella")
    media = MediaFileUpload(str(path), resumable=True)
    if files:
        return service.files().update(
            fileId=files[0]["id"], media_body=media, fields="id,name"
        ).execute()
    return service.files().create(
        body={"name": path.name, "parents": [folder_id]},
        media_body=media, fields="id,name",
    ).execute()


def download_output_artifact(filename, destination):
    # Permette di misurare nuovi runtime partendo dal QAT già salvato, senza riaddestrare.
    destination = Path(destination)
    if destination.is_file():
        return destination
    service = drive_service()
    folder_id = resolve_output_folder(service)
    escaped_folder_id = drive_query_escape(folder_id)
    escaped_filename = drive_query_escape(filename)
    response = service.files().list(
        q=f"'{escaped_folder_id}' in parents and name = '{escaped_filename}' and trashed = false",
        fields="files(id,name)", pageSize=10,
    ).execute()
    files = response.get("files", [])
    if len(files) != 1:
        raise FileNotFoundError(
            f"Atteso un solo {filename} in {QAT_DRIVE_FOLDER_NAME}, trovati {len(files)}"
        )
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=files[0]["id"])
    with io.FileIO(destination, "wb") as stream:
        downloader = MediaIoBaseDownload(stream, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return destination


source_checkpoint_path = download_checkpoint()
RESTORED_RESULTS = None
if RESTORE_SAVED_QAT:
    download_output_artifact(Path(BEST_QAT_PATH).name, BEST_QAT_PATH)
    # Conserva storia, pre/post QAT e metriche già salvate quando aggiungiamo il confronto runtime.
    try:
        download_output_artifact(Path(RESULTS_PATH).name, RESULTS_PATH)
        RESTORED_RESULTS = json.loads(Path(RESULTS_PATH).read_text(encoding="utf-8"))
    except FileNotFoundError:
        print("results.json precedente non trovato: verrà creato senza storia pregressa")
print(source_checkpoint_path)

## 5. Dataset senza oversampling e senza augmentation

Resize e normalizzazione coincidono con il training FP32, ma QAT, calibrazione e valutazione usano la distribuzione naturale: nessun oversampling e nessuna augmentation. I 500 esempi di calibrazione sono estratti dal train con seed fisso; validation e test non partecipano alla calibrazione.

In [ ]:
class SegmentationTransform:
    def __init__(self):
        self.mean = torch.tensor(IMAGE_MEAN).view(3, 1, 1)
        self.std = torch.tensor(IMAGE_STD).view(3, 1, 1)

    def __call__(self, image, mask):
        # Resize deterministico: nessuna augmentation viene applicata.
        size = (IMAGE_SIZE[1], IMAGE_SIZE[0])
        image = image.resize(size, Image.Resampling.BILINEAR)
        mask = mask.resize(size, Image.Resampling.NEAREST)
        image = torch.from_numpy(np.asarray(image, dtype=np.float32).copy())
        image = image.permute(2, 0, 1).div_(255.0)
        image = (image - self.mean) / self.std
        mask = torch.from_numpy(np.asarray(mask, dtype=np.int64).copy()).long()
        return image, mask


class S5MarsDataset(Dataset):
    def __init__(self, split, max_samples=None):
        token = get_secret("hf_token", env_name="HF_TOKEN", required=False)
        dataset = load_dataset(REPO_ID, split=split, token=token)
        if max_samples is not None:
            rng = np.random.default_rng(SEED)
            indices = rng.permutation(len(dataset))[:min(max_samples, len(dataset))]
            dataset = dataset.select(indices.tolist())
        self.dataset = dataset
        self.transform = SegmentationTransform()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        sample = self.dataset[index]
        image, mask = self.transform(
            sample["image"].convert("RGB"), sample["mask"].convert("L")
        )
        return {"image": image, "mask": mask}


def make_loader(dataset, batch_size, shuffle):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        drop_last=False,
        generator=torch.Generator().manual_seed(SEED),
    )


train_dataset = S5MarsDataset(TRAIN_SPLIT, MAX_TRAIN_SAMPLES)
val_dataset = S5MarsDataset(VAL_SPLIT, MAX_VAL_SAMPLES)
train_loader = make_loader(train_dataset, TRAIN_BATCH_SIZE, shuffle=True)
val_loader = make_loader(val_dataset, EVAL_BATCH_SIZE, shuffle=False)

# La calibrazione iniziale usa 500 campioni train casuali ma riproducibili.
rng = np.random.default_rng(SEED)
calibration_indices = rng.permutation(len(train_dataset))[:min(CALIBRATION_SAMPLES, len(train_dataset))]
calibration_dataset = Subset(train_dataset, calibration_indices.tolist())
calibration_loader = make_loader(calibration_dataset, CALIBRATION_BATCH_SIZE, shuffle=False)
print("train", len(train_dataset), "val", len(val_dataset), "calibration", len(calibration_dataset))

## 6. Modello FP32 e loss

Il modello viene ricostruito dalla configurazione SegFormer e poi popolato integralmente dal checkpoint, con gate su architettura, risoluzione e testa a nove classi. La loss combina cross-entropy e Generalized Dice square; il teacher FP32 resta congelato per la distillazione.

In [ ]:
class MarsSegmentationModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Scarica soltanto la configurazione B0: i pesi ADE20K hanno 150 classi e non servono.
        config = SegformerConfig.from_pretrained(
            PRETRAINED_NAME,
            num_labels=NUM_CLASSES,
            semantic_loss_ignore_index=IGNORE_INDEX,
            id2label=CLASS_NAMES,
            label2id={label: index for index, label in CLASS_NAMES.items()},
        )
        self.model = SegformerForSemanticSegmentation(config)

    def forward(self, images):
        # Hugging Face produce logits a 1/4 della risoluzione; il contratto S5Mars è 512x512.
        # La tupla evita di esportare il dataclass ModelOutput con il tracer TorchScript.
        logits = self.model(pixel_values=images, return_dict=False)[0]
        return F.interpolate(
            logits, size=images.shape[-2:], mode="bilinear", align_corners=False
        )


def load_fp32_model(path):
    checkpoint = torch.load(path, map_location="cpu")
    config = checkpoint.get("config", {}) if isinstance(checkpoint, dict) else {}
    expected = {
        "model_name": MODEL_NAME,
        "pretrained_name": PRETRAINED_NAME,
    }
    mismatches = {
        key: {"expected": value, "checkpoint": config.get(key)}
        for key, value in expected.items()
        if config.get(key) not in (None, value)
    }
    checkpoint_image_size = config.get("image_size")
    if checkpoint_image_size is not None and tuple(checkpoint_image_size) != tuple(IMAGE_SIZE):
        mismatches["image_size"] = {
            "expected": list(IMAGE_SIZE), "checkpoint": checkpoint_image_size
        }
    if mismatches:
        raise ValueError(f"Checkpoint incompatibile: {mismatches}")
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    classifier_key = "model.decode_head.classifier.weight"
    if classifier_key not in state_dict or state_dict[classifier_key].shape[0] != NUM_CLASSES:
        found = None if classifier_key not in state_dict else state_dict[classifier_key].shape[0]
        raise ValueError(f"Testa checkpoint incompatibile: attese {NUM_CLASSES} classi, trovate {found}")
    model = MarsSegmentationModel()
    model.load_state_dict(state_dict, strict=True)
    if model.model.decode_head.classifier.out_channels != NUM_CLASSES:
        raise RuntimeError("La testa SegFormer caricata non ha 9 canali di output")
    return model


class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.cross_entropy = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    def forward(self, logits, targets):
        valid = targets != IGNORE_INDEX
        safe_targets = targets.masked_fill(~valid, 0)
        probabilities = torch.softmax(logits, dim=1) * valid.unsqueeze(1)
        one_hot = F.one_hot(safe_targets, NUM_CLASSES).permute(0, 3, 1, 2).float()
        one_hot = one_hot * valid.unsqueeze(1)
        volumes = one_hot.sum((0, 2, 3))
        weights = torch.where(volumes > 0, volumes.clamp_min(1).pow(-2), 0.0)
        intersection = (probabilities * one_hot).sum((0, 2, 3))
        denominator = (probabilities + one_hot).sum((0, 2, 3))
        dice = 1.0 - (
            2.0 * (weights * intersection).sum() + DICE_SMOOTH
        ) / ((weights * denominator).sum() + DICE_SMOOTH)
        return (
            COMBINED_LOSS_ALPHA * self.cross_entropy(logits, targets)
            + (1.0 - COMBINED_LOSS_ALPHA) * dice
        )


def freeze_batch_norm(model):
    # Le statistiche BN restano quelle del training FP32; si aggiornano solo i pesi QAT.
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            module.requires_grad_(False)


teacher = load_fp32_model(source_checkpoint_path).to(DEVICE).eval()
student = copy.deepcopy(teacher)
teacher.requires_grad_(False)  # Il teacher produce soltanto i target di distillazione.
student.requires_grad_(True)   # Lo student deve invece aggiornare i pesi durante il QAT.
criterion = CombinedLoss()
print("parameters", sum(parameter.numel() for parameter in teacher.parameters()))

## 7. Metriche

PyTorch e TensorRT condividono la stessa matrice di confusione e le stesse metriche. Poiché `IGNORE_INDEX=-100` non compare nelle mask 0..8, Background e tutte le altre classi entrano in pixel accuracy, mIoU e IoU per classe.

In [ ]:
def update_confusion(confusion, predictions, targets):
    predictions = predictions.reshape(-1).cpu()
    targets = targets.reshape(-1).cpu()
    valid = (targets != IGNORE_INDEX) & (targets >= 0) & (targets < NUM_CLASSES)
    indices = targets[valid] * NUM_CLASSES + predictions[valid]
    confusion += torch.bincount(
        indices, minlength=NUM_CLASSES * NUM_CLASSES
    ).reshape(NUM_CLASSES, NUM_CLASSES)


def segmentation_metrics(confusion):
    confusion = confusion.double()
    true_positive = confusion.diag()
    support = confusion.sum(1)
    union = support + confusion.sum(0) - true_positive
    iou = torch.full((NUM_CLASSES,), float("nan"), dtype=torch.float64)
    present = union > 0
    iou[present] = true_positive[present] / union[present]
    valid_classes = support > 0
    if 0 <= IGNORE_INDEX < NUM_CLASSES:
        valid_classes[IGNORE_INDEX] = False
    return {
        "pixel_accuracy": (true_positive.sum() / confusion.sum().clamp_min(1)).item(),
        "miou": iou[valid_classes].mean().item(),
        "per_class_iou": {
            CLASS_NAMES[index]: (iou[index].item() if present[index] else None)
            for index in range(NUM_CLASSES)
        },
    }


@torch.inference_mode()
def evaluate_torch(model, loader, split):
    model.eval()
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.int64)
    for batch in tqdm(loader, desc=f"evaluate {split}"):
        images = batch["image"].to(DEVICE, non_blocking=True)
        predictions = model(images).argmax(1)
        update_confusion(confusion, predictions, batch["mask"])
    metrics = segmentation_metrics(confusion)
    metrics.update(split=split, samples=len(loader.dataset))
    return metrics


RESULTS = {
    "experiment": {
        "method": "nvidia_modelopt_qat",
        "model": EXPERIMENT,
        "quantization": "INT8 Conv2d + encoder MLP Linear + decoder projection Linear; attention e altri operatori FP16",
        "variant": QAT_VARIANT,
        "fp16_fallback_module_prefixes": list(FP16_FALLBACK_MODULE_PREFIXES),
        "fp16_fallback_reason": None,
        "calibration_samples": len(calibration_dataset),
        "no_oversampling": True,
        "no_augmentation": True,
        "test_is_final_holdout": True,
    },
    "training": {},
    "artifacts": {},
    "benchmarks": {},
    "comparisons": {},
}
if RESTORE_SAVED_QAT and RESTORED_RESULTS is not None:
    fresh_results = RESULTS
    RESULTS = RESTORED_RESULTS
    RESULTS.setdefault("experiment", {}).update(fresh_results["experiment"])
    for section in ("training", "artifacts", "benchmarks", "comparisons"):
        RESULTS.setdefault(section, {})


def save_results():
    Path(RESULTS_PATH).write_text(json.dumps(RESULTS, indent=2), encoding="utf-8")


RESULTS["benchmarks"]["pytorch_fp32_cuda"] = {
    "runtime": "pytorch",
    "precision": "fp32",
    "metrics": evaluate_torch(teacher, val_loader, VAL_SPLIT),
}
save_results()
print(json.dumps(RESULTS["benchmarks"]["pytorch_fp32_cuda"], indent=2))

## 8. Preparazione ModelOpt INT8 selettiva: Conv + MLP + decoder

ModelOpt parte da una copia pulita del teacher. Tutti i quantizer sono disabilitati, poi vengono abilitati soltanto i Conv2d e i gruppi Linear dichiarati. Nomi, conteggi e stato di ogni quantizer sono verificati con gate bloccanti per intercettare cambiamenti di Transformers o ModelOpt.

In [ ]:
# La cella può essere rieseguita: riparte sempre dallo stesso teacher FP32 pulito.
if "student" in globals():
    del student
gc.collect()
torch.cuda.empty_cache()
student = copy.deepcopy(teacher).to(DEVICE)
student.requires_grad_(True)


def segformer_linear_group(name):
    # Supporta sia i nomi Transformers storici sia quelli più recenti.
    if name.endswith((".mlp.dense1", ".mlp.dense2", ".mlp.fc1", ".mlp.fc2")):
        return "encoder_mlp"
    decoder_projection = (
        ".decode_head." in name
        and name.endswith(".proj")
        and (".linear_c." in name or ".linear_projections." in name)
    )
    return "decoder_projection" if decoder_projection else None


source_linear_modules = {
    name: module for name, module in student.named_modules() if isinstance(module, nn.Linear)
}
discovered_linear_groups = {
    group: sorted(name for name in source_linear_modules if segformer_linear_group(name) == group)
    for group in INT8_LINEAR_GROUPS
}
expected_group_counts = {"encoder_mlp": 16, "decoder_projection": 4}
actual_group_counts = {group: len(names) for group, names in discovered_linear_groups.items()}
if actual_group_counts != expected_group_counts:
    raise RuntimeError({
        "reason": "Linear SegFormer selezionati non riconosciuti",
        "expected_group_counts": expected_group_counts,
        "actual_group_counts": actual_group_counts,
        "all_linear_module_names": sorted(source_linear_modules),
    })
target_linear_module_names = sorted(
    name
    for names in discovered_linear_groups.values()
    for name in names
    if not any(name == prefix or name.startswith(prefix + ".")
               for prefix in FP16_FALLBACK_MODULE_PREFIXES)
)

# Tutto parte disabilitato; abilitiamo soltanto Conv e i 20 Linear scelti.
conv_parent_class = mtq.QuantModuleRegistry.get_key(nn.Conv2d)
linear_parent_class = mtq.QuantModuleRegistry.get_key(nn.Linear)
if not conv_parent_class or not linear_parent_class:
    raise RuntimeError({"conv": conv_parent_class, "linear": linear_parent_class})
linear_quantizer_cfg = []
for name in target_linear_module_names:
    linear_quantizer_cfg.extend([
        {
            "quantizer_name": f"{name}.weight_quantizer",
            "parent_class": linear_parent_class,
            "enable": True,
            "cfg": {"num_bits": 8, "axis": 0, "trt_high_precision_dtype": "Half"},
        },
        {
            "quantizer_name": f"{name}.input_quantizer",
            "parent_class": linear_parent_class,
            "enable": True,
            "cfg": {"num_bits": 8, "axis": None, "trt_high_precision_dtype": "Half"},
        },
    ])
INT8_SELECTIVE_CFG = {
    "quant_cfg": [
        {"quantizer_name": "*", "enable": False},
        {
            "quantizer_name": "*weight_quantizer",
            "parent_class": conv_parent_class,
            "enable": True,
            "cfg": {"num_bits": 8, "axis": 0, "trt_high_precision_dtype": "Half"},
        },
        {
            "quantizer_name": "*input_quantizer",
            "parent_class": conv_parent_class,
            "enable": True,
            "cfg": {"num_bits": 8, "axis": None, "trt_high_precision_dtype": "Half"},
        },
    ] + linear_quantizer_cfg + [
        {"quantizer_name": f"*{prefix}.*", "enable": False}
        for prefix in FP16_FALLBACK_MODULE_PREFIXES
    ],
    "algorithm": "max",
}


@torch.inference_mode()
def calibration_loop(model):
    model.eval()
    for batch in tqdm(calibration_loader, desc="ModelOpt calibration"):
        model(batch["image"].to(DEVICE, non_blocking=True))


if RESTORE_SAVED_QAT:
    # mto.restore ricrea quantizer, scale e pesi dal checkpoint prodotto da mto.save.
    student = mto.restore(student, BEST_QAT_PATH, map_location=DEVICE).to(DEVICE)
else:
    # ModelOpt inserisce fake quantizer soltanto nei Conv e nei Linear selezionati.
    student = mtq.quantize(student, INT8_SELECTIVE_CFG, forward_loop=calibration_loop)
freeze_batch_norm(student)

modules = dict(student.named_modules())
conv_modules = {
    name: module for name, module in modules.items() if isinstance(module, nn.Conv2d)
}
linear_modules = {
    name: module for name, module in modules.items() if isinstance(module, nn.Linear)
}
layer_norm_modules = {
    name: module for name, module in modules.items() if isinstance(module, nn.LayerNorm)
}
if not conv_modules or not linear_modules or not layer_norm_modules:
    raise RuntimeError({
        "conv2d": len(conv_modules),
        "linear": len(linear_modules),
        "layer_norm": len(layer_norm_modules),
        "reason": "struttura SegFormer non riconosciuta",
    })
expected_fp16_conv_names = sorted(
    name for name in conv_modules
    if any(name == prefix or name.startswith(prefix + ".")
           for prefix in FP16_FALLBACK_MODULE_PREFIXES)
)
expected_int8_conv_names = sorted(set(conv_modules) - set(expected_fp16_conv_names))
expected_int8_linear_names = sorted(target_linear_module_names)
expected_fp16_linear_names = sorted(set(linear_modules) - set(expected_int8_linear_names))
int8_conv_names, fp16_fallback_conv_names, partial_conv_names = [], [], []
for name, module in conv_modules.items():
    # I Conv in fallback conservano i quantizer ModelOpt, ma entrambi disabilitati.
    input_quantizer = getattr(module, "input_quantizer", None)
    weight_quantizer = getattr(module, "weight_quantizer", None)
    input_enabled = input_quantizer is not None and input_quantizer.is_enabled
    weight_enabled = weight_quantizer is not None and weight_quantizer.is_enabled
    if input_enabled and weight_enabled:
        int8_conv_names.append(name)
    elif not input_enabled and not weight_enabled:
        fp16_fallback_conv_names.append(name)
    else:
        partial_conv_names.append(name)

int8_linear_names, fp16_linear_names, partial_linear_names = [], [], []
for name, module in linear_modules.items():
    input_quantizer = getattr(module, "input_quantizer", None)
    weight_quantizer = getattr(module, "weight_quantizer", None)
    input_enabled = input_quantizer is not None and input_quantizer.is_enabled
    weight_enabled = weight_quantizer is not None and weight_quantizer.is_enabled
    if input_enabled and weight_enabled:
        int8_linear_names.append(name)
    elif not input_enabled and not weight_enabled:
        fp16_linear_names.append(name)
    else:
        partial_linear_names.append(name)

enabled_quantizer_names = sorted(
    name for name, module in modules.items()
    if isinstance(module, TensorQuantizer) and module.is_enabled
)
expected_enabled_quantizer_names = sorted(
    f"{module_name}.{quantizer_name}"
    for module_name in expected_int8_conv_names + expected_int8_linear_names
    for quantizer_name in ("input_quantizer", "weight_quantizer")
)
unexpected_enabled_quantizers = sorted(
    set(enabled_quantizer_names) - set(expected_enabled_quantizer_names)
)
missing_enabled_quantizers = sorted(
    set(expected_enabled_quantizer_names) - set(enabled_quantizer_names)
)

int8_conv_names.sort()
fp16_fallback_conv_names.sort()
int8_linear_names.sort()
fp16_linear_names.sort()
conv_count = len(conv_modules)
int8_conv_count = len(int8_conv_names)
int8_linear_count = len(int8_linear_names)
if (
    partial_conv_names
    or partial_linear_names
    or int8_conv_names != expected_int8_conv_names
    or fp16_fallback_conv_names != expected_fp16_conv_names
    or int8_linear_names != expected_int8_linear_names
    or fp16_linear_names != expected_fp16_linear_names
    or unexpected_enabled_quantizers
    or missing_enabled_quantizers
):
    raise RuntimeError({
        "partial_conv_quantization": partial_conv_names,
        "partial_linear_quantization": partial_linear_names,
        "unexpected_enabled_quantizers": unexpected_enabled_quantizers,
        "missing_enabled_quantizers": missing_enabled_quantizers,
        "expected_int8_conv": expected_int8_conv_names,
        "actual_int8_conv": int8_conv_names,
        "expected_fp16_conv": expected_fp16_conv_names,
        "actual_fp16_conv": fp16_fallback_conv_names,
        "expected_int8_linear": expected_int8_linear_names,
        "actual_int8_linear": int8_linear_names,
        "expected_fp16_linear": expected_fp16_linear_names,
        "actual_fp16_linear": fp16_linear_names,
    })

RESULTS["experiment"]["modelopt_quantizer_audit"] = {
    "conv2d_total": conv_count,
    "conv2d_int8": int8_conv_count,
    "conv2d_fp16_fallback": len(fp16_fallback_conv_names),
    "fp16_fallback_conv_names": fp16_fallback_conv_names,
    "linear_total": len(linear_modules),
    "linear_int8": int8_linear_count,
    "linear_int8_names": int8_linear_names,
    "linear_fp16": len(fp16_linear_names),
    "linear_fp16_names": fp16_linear_names,
    "linear_groups_int8": discovered_linear_groups,
    "layer_norm_total_fp16": len(layer_norm_modules),
    "enabled_quantizer_names": enabled_quantizer_names,
}
if not RESTORE_SAVED_QAT:
    RESULTS["benchmarks"]["pytorch_modelopt_initialized_fake_quant"] = {
        "runtime": "pytorch",
        "precision": "fake_quant_int8_conv_mlp_decoder_linear",
        "metrics": evaluate_torch(student, val_loader, VAL_SPLIT),
    }
save_results()
print(json.dumps(RESULTS["experiment"]["modelopt_quantizer_audit"], indent=2))

## 9. Quantization Aware Training

Lo student fake-quant è ottimizzato con loss supervisionata e distillazione dal teacher FP32, AMP, gradient accumulation, clipping, warm-up e cosine decay. Il best checkpoint dipende soltanto dalla mIoU di validation; early stopping e upload non consultano il test.

In [ ]:
def distillation_loss(student_logits, teacher_logits):
    temperature = DISTILLATION_TEMPERATURE
    student_log_probs = F.log_softmax(student_logits / temperature, dim=1)
    teacher_probs = F.softmax(teacher_logits / temperature, dim=1)
    # Media per pixel: la scala della KL non dipende dalla risoluzione 512x512.
    return F.kl_div(student_log_probs, teacher_probs, reduction="none").sum(1).mean() * temperature**2


trainable_parameters = [
    parameter for parameter in student.parameters() if parameter.requires_grad
]
if not trainable_parameters:
    raise RuntimeError("Lo student non contiene parametri addestrabili")
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
optimizer_steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
total_steps = max(EPOCHS * optimizer_steps_per_epoch, 1)
warmup_steps = WARMUP_EPOCHS * optimizer_steps_per_epoch

def lr_multiplier(step):
    if warmup_steps and step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1.0 + math.cos(math.pi * min(max(progress, 0.0), 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
history = []
restored_validation = (
    evaluate_torch(student, val_loader, VAL_SPLIT) if RESTORE_SAVED_QAT else None
)
best_miou = restored_validation["miou"] if RESTORE_SAVED_QAT else -float("inf")
best_weights = (
    {name: value.detach().cpu().clone() for name, value in student.state_dict().items()}
    if RESTORE_SAVED_QAT else None
)
epochs_without_improvement = 0
service = None
drive_folder_id = None

for epoch in range(0 if RESTORE_SAVED_QAT else EPOCHS):
    student.train()
    freeze_batch_norm(student)
    optimizer.zero_grad(set_to_none=True)
    running = {"loss": 0.0, "supervised": 0.0, "distillation": 0.0}

    for batch_index, batch in enumerate(tqdm(train_loader, desc=f"QAT {epoch + 1}/{EPOCHS}")):
        images = batch["image"].to(DEVICE, non_blocking=True)
        masks = batch["mask"].to(DEVICE, non_blocking=True)

        # Il teacher resta FP32 e non riceve gradienti.
        with torch.no_grad():
            teacher_logits = teacher(images)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            student_logits = student(images)
        supervised = criterion(student_logits.float(), masks)
        distillation = distillation_loss(student_logits.float(), teacher_logits.float())
        loss = SUPERVISED_WEIGHT * supervised + DISTILLATION_WEIGHT * distillation
        scaler.scale(loss / GRADIENT_ACCUMULATION_STEPS).backward()

        should_step = (
            (batch_index + 1) % GRADIENT_ACCUMULATION_STEPS == 0
            or batch_index + 1 == len(train_loader)
        )
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running["loss"] += loss.item()
        running["supervised"] += supervised.item()
        running["distillation"] += distillation.item()

    val_metrics = evaluate_torch(student, val_loader, VAL_SPLIT)
    row = {
        "epoch": epoch + 1,
        "lr": optimizer.param_groups[0]["lr"],
        "train_loss": running["loss"] / len(train_loader),
        "train_supervised_loss": running["supervised"] / len(train_loader),
        "train_distillation_loss": running["distillation"] / len(train_loader),
        "val_pixel_accuracy": val_metrics["pixel_accuracy"],
        "val_miou": val_metrics["miou"],
    }
    history.append(row)
    print(json.dumps(row, indent=2))

    # mto.save conserva insieme pesi e stato dei quantizer ModelOpt.
    mto.save(student, LAST_QAT_PATH)
    if val_metrics["miou"] > best_miou:
        best_miou = val_metrics["miou"]
        epochs_without_improvement = 0
        best_weights = {
            name: value.detach().cpu().clone()
            for name, value in student.state_dict().items()
        }
        mto.save(student, BEST_QAT_PATH)
        if UPLOAD_BEST_EACH_EPOCH:
            try:
                service = service or drive_service()
                drive_folder_id = drive_folder_id or resolve_output_folder(service)
                upload_file(BEST_QAT_PATH, service, drive_folder_id)
            except Exception as error:
                print("Upload best rinviato:", error)
    else:
        epochs_without_improvement += 1

    Path(HISTORY_PATH).write_text(json.dumps(history, indent=2), encoding="utf-8")
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping")
        break

if best_weights is None:
    raise RuntimeError("Nessun checkpoint QAT prodotto")
student.load_state_dict(best_weights, strict=True)
student.eval()
freeze_batch_norm(student)

if RESTORE_SAVED_QAT:
    RESULTS.setdefault("training", {})["runtime_comparison_restore"] = {
        "restored_without_retraining": True,
        "best_val_miou_recomputed": best_miou,
        "best_checkpoint": BEST_QAT_PATH,
    }
else:
    RESULTS["training"] = {
        "epochs_completed": len(history),
        "best_epoch": max(history, key=lambda row: row["val_miou"])["epoch"],
        "best_val_miou": best_miou,
        "history": history,
        "best_checkpoint": BEST_QAT_PATH,
    }
qat_val_metrics = evaluate_torch(student, val_loader, VAL_SPLIT)
RESULTS["benchmarks"]["pytorch_modelopt_qat_fake_quant"] = {
    "runtime": "pytorch",
    "precision": "fake_quant_int8_conv_mlp_decoder_linear_qat",
    "metrics": qat_val_metrics,
}
if not RESTORE_SAVED_QAT:
    initialized_val_metrics = RESULTS["benchmarks"]["pytorch_modelopt_initialized_fake_quant"]["metrics"]
    pre_post_qat = {
        "split": VAL_SPLIT,
        "before_qat": {
            "pixel_accuracy": initialized_val_metrics["pixel_accuracy"],
            "miou": initialized_val_metrics["miou"],
        },
        "after_qat": {
            "pixel_accuracy": qat_val_metrics["pixel_accuracy"],
            "miou": qat_val_metrics["miou"],
        },
        "delta_after_minus_before": {
            "pixel_accuracy": qat_val_metrics["pixel_accuracy"] - initialized_val_metrics["pixel_accuracy"],
            "miou": qat_val_metrics["miou"] - initialized_val_metrics["miou"],
        },
    }
    RESULTS["training"]["pre_post_qat_validation"] = pre_post_qat
save_results()
print(json.dumps(RESULTS["training"], indent=2))

## 10. Export ONNX Q/DQ e folding dei pesi costanti

L'export conserva I/O FP32 e fallback interno FP16. Il folding sostituisce soltanto i pattern costanti peso→Q→DQ con initializer INT8→DQ equivalenti; le Q/DQ delle attivazioni restano intatte. Checker ONNX, audit di copertura e parità su CUDA EP sono obbligatori prima di TensorRT.

In [ ]:
def fold_constant_int8_weights(model):
    # Converte: initializer float -> Q -> DQ -> peso Conv/MatMul/Gemm.
    # Le Q/DQ delle attivazioni restano intatte, quindi il calcolo pesato resta INT8.
    initializers = {item.name: item for item in model.graph.initializer}
    producers = {output: node for node in model.graph.node for output in node.output}
    consumers = {}
    for node in model.graph.node:
        for input_index, name in enumerate(node.input):
            consumers.setdefault(name, []).append((node, input_index))

    def constant_array(name):
        if name in initializers:
            return numpy_helper.to_array(initializers[name])
        producer = producers.get(name)
        if producer is None:
            return None
        if producer.op_type == "Constant":
            value = next((attr for attr in producer.attribute if attr.name == "value"), None)
            return None if value is None else numpy_helper.to_array(value.t)
        if producer.op_type == "Cast":
            source = constant_array(producer.input[0])
            target = next(
                (onnx.helper.get_attribute_value(attr) for attr in producer.attribute if attr.name == "to"),
                None,
            )
            if source is None or target is None:
                return None
            return source.astype(onnx.helper.tensor_dtype_to_np_dtype(target))
        return None

    def downstream_weighted_users(tensor_name):
        # I Linear 3D possono esportare DQ -> Transpose -> MatMul.
        # Si attraversano soltanto operatori di layout che non cambiano i valori del peso.
        found = {"conv": {}, "linear": {}}
        pending = [tensor_name]
        visited = set()
        while pending:
            current = pending.pop()
            if current in visited:
                continue
            visited.add(current)
            for user, input_index in consumers.get(current, []):
                key = user.name or f"{user.op_type}:{user.output[0]}"
                if user.op_type == "Conv" and input_index == 1:
                    found["conv"][key] = user
                elif user.op_type in {"MatMul", "Gemm"} and input_index == 1:
                    found["linear"][key] = user
                elif input_index == 0 and user.op_type in {"Transpose", "Reshape", "Cast", "Identity"}:
                    pending.extend(user.output)
        return list(found["conv"].values()), list(found["linear"].values())

    folded_outputs = set()
    quantized_initializers = []
    folded_conv_names = []
    folded_linear_names = []
    issues = []
    for node in model.graph.node:
        if node.op_type != "QuantizeLinear" or node.input[0] not in initializers:
            continue
        dq_users = consumers.get(node.output[0], [])
        if not dq_users or any(user.op_type != "DequantizeLinear" for user, _ in dq_users):
            continue

        conv_users_by_name = {}
        linear_users_by_name = {}
        for dq_node, _ in dq_users:
            conv_users, linear_users = downstream_weighted_users(dq_node.output[0])
            conv_users_by_name.update({
                node.name or f"{node.op_type}:{node.output[0]}": node for node in conv_users
            })
            linear_users_by_name.update({
                node.name or f"{node.op_type}:{node.output[0]}": node for node in linear_users
            })
        conv_users = list(conv_users_by_name.values())
        linear_users = list(linear_users_by_name.values())
        if not conv_users and not linear_users:
            continue

        scale_array = constant_array(node.input[1])
        zero_array = constant_array(node.input[2]) if len(node.input) > 2 else None
        if scale_array is None or zero_array is None:
            issues.append({"quantizer": node.name, "reason": "scale_or_zero_point_not_constant"})
            continue
        weight_array = numpy_helper.to_array(initializers[node.input[0]]).astype(np.float32)
        scale_array = np.asarray(scale_array, dtype=np.float32)
        zero_array = np.asarray(zero_array)
        if zero_array.dtype != np.int8 or np.any(zero_array != 0):
            issues.append({"quantizer": node.name, "reason": "non_symmetric_int8_zero_point"})
            continue
        if np.any(scale_array <= 0):
            issues.append({"quantizer": node.name, "reason": "non_positive_scale"})
            continue

        axis = next(
            (onnx.helper.get_attribute_value(attr) for attr in node.attribute if attr.name == "axis"),
            1,
        )
        axis = axis if axis >= 0 else weight_array.ndim + axis
        if not 0 <= axis < weight_array.ndim:
            issues.append({"quantizer": node.name, "reason": "invalid_axis", "axis": axis})
            continue
        if scale_array.size > 1 and (
            scale_array.size != weight_array.shape[axis] or zero_array.size != scale_array.size
        ):
            issues.append({"quantizer": node.name, "reason": "per_channel_shape_mismatch"})
            continue
        if scale_array.size == 1:
            scale_view = scale_array.reshape([1] * weight_array.ndim)
            zero_view = zero_array.reshape([1] * weight_array.ndim)
        else:
            broadcast_shape = [1] * weight_array.ndim
            broadcast_shape[axis] = scale_array.size
            scale_view = scale_array.reshape(broadcast_shape)
            zero_view = zero_array.reshape(broadcast_shape)

        # np.rint implementa round-to-nearest ties-to-even, come QuantizeLinear INT8.
        quantized = np.rint(weight_array / scale_view) + zero_view
        quantized = np.clip(quantized, -128, 127).astype(np.int8)
        quantized_initializers.append(numpy_helper.from_array(quantized, node.output[0]))
        folded_outputs.add(node.output[0])
        folded_conv_names.extend(conv.name for conv in conv_users)
        folded_linear_names.extend(linear.name for linear in linear_users)

    kept_nodes = [
        node for node in model.graph.node
        if node.op_type != "QuantizeLinear" or node.output[0] not in folded_outputs
    ]
    del model.graph.node[:]
    model.graph.node.extend(kept_nodes)
    model.graph.initializer.extend(quantized_initializers)

    # Elimina soltanto initializer rimasti senza consumatori dopo il folding.
    used_inputs = {name for graph_node in model.graph.node for name in graph_node.input}
    used_inputs.update(output.name for output in model.graph.output)
    kept_initializers = [item for item in model.graph.initializer if item.name in used_inputs]
    del model.graph.initializer[:]
    model.graph.initializer.extend(kept_initializers)
    return model, {
        "folded_weight_quantizers": len(folded_outputs),
        "int8_weight_initializers": len(quantized_initializers),
        "folded_conv_names": folded_conv_names,
        "folded_linear_names": folded_linear_names,
        "issues": issues,
    }


sample = next(iter(val_loader))["image"].to(DEVICE)
student.eval()
# Il tipo influisce solo sull'export: si può rieseguire questa cella senza ripetere il QAT.
for module in student.modules():
    if isinstance(module, TensorQuantizer) and module.is_enabled:
        module.trt_high_precision_dtype = "Half"
with torch.inference_mode():
    fp32_fake_quant_output = student(sample).detach().cpu().numpy()


class FP32IOFP16Fallback(nn.Module):
    # Mantiene I/O FP32, mentre tutto il calcolo non INT8 interno usa FP16.
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, images):
        return self.model(images.to(torch.float16)).float()


# Una copia evita di alterare lo student QAT usato per checkpoint e metriche PyTorch.
export_model = FP32IOFP16Fallback(copy.deepcopy(student).half()).eval()
with torch.inference_mode():
    fp16_fallback_output = export_model(sample).detach().cpu().numpy()
fallback_parity = {
    "max_abs_error": float(np.max(np.abs(fp16_fallback_output - fp32_fake_quant_output))),
    "mean_abs_error": float(np.mean(np.abs(fp16_fallback_output - fp32_fake_quant_output))),
    "prediction_agreement": float(np.mean(
        fp16_fallback_output.argmax(1) == fp32_fake_quant_output.argmax(1)
    )),
}
if fallback_parity["prediction_agreement"] < FP16_FALLBACK_MIN_PREDICTION_AGREEMENT:
    raise RuntimeError(f"Fallback FP16 non fedele al fake-quant FP32: {fallback_parity}")

# ModelOpt registra i symbolic Q/DQ per l'exporter TorchScript; per questo dynamo=False.
with torch.inference_mode():
    torch.onnx.export(
        export_model,
        sample,
        ONNX_PATH,
        input_names=["images"],
        output_names=["logits"],
        opset_version=ONNX_OPSET,
        dynamo=False,
        do_constant_folding=True,
    )

onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model, full_check=True)
raw_conv_count = sum(node.op_type == "Conv" for node in onnx_model.graph.node)
onnx_model, weight_fold = fold_constant_int8_weights(onnx_model)
if raw_conv_count != conv_count:
    raise RuntimeError({"pytorch_conv": conv_count, "onnx_conv": raw_conv_count})
if (
    weight_fold["issues"]
    or weight_fold["folded_weight_quantizers"] != int8_conv_count + int8_linear_count
):
    raise RuntimeError({
        "expected_int8_weight_initializers": int8_conv_count + int8_linear_count,
        "weight_fold": weight_fold,
    })
onnx.checker.check_model(onnx_model, full_check=True)
onnx.save(onnx_model, ONNX_PATH)
producers = {output: node for node in onnx_model.graph.node for output in node.output}
conv_nodes = [node for node in onnx_model.graph.node if node.op_type == "Conv"]
linear_onnx_nodes = [
    node for node in onnx_model.graph.node if node.op_type in {"MatMul", "Gemm"}
]
q_nodes = [node for node in onnx_model.graph.node if node.op_type == "QuantizeLinear"]
dq_nodes = [node for node in onnx_model.graph.node if node.op_type == "DequantizeLinear"]
def traces_to_dequantize(tensor_name):
    # Accetta il layout DQ -> Transpose -> MatMul prodotto dai Linear 3D.
    visited = set()
    while tensor_name not in visited:
        visited.add(tensor_name)
        producer = producers.get(tensor_name)
        if producer is None:
            return False
        if producer.op_type == "DequantizeLinear":
            return True
        if producer.op_type not in {"Transpose", "Reshape", "Flatten", "Cast", "Identity"}:
            return False
        tensor_name = producer.input[0]
    return False


def is_qdq_wrapped_weighted_op(node):
    return (
        len(node.input) >= 2
        and traces_to_dequantize(node.input[0])
        and traces_to_dequantize(node.input[1])
    )

wrapped_conv_nodes = [node for node in conv_nodes if is_qdq_wrapped_weighted_op(node)]
fp16_onnx_conv_nodes = [node for node in conv_nodes if not is_qdq_wrapped_weighted_op(node)]
unexpected_fp16_nodes = [
    node.name for node in fp16_onnx_conv_nodes
    if not any(prefix in node.name.replace("/", ".")
               for prefix in FP16_FALLBACK_MODULE_PREFIXES)
]
unexpected_int8_nodes = [
    node.name for node in wrapped_conv_nodes
    if any(prefix in node.name.replace("/", ".")
           for prefix in FP16_FALLBACK_MODULE_PREFIXES)
]
wrapped_linear_nodes = [
    node for node in linear_onnx_nodes if is_qdq_wrapped_weighted_op(node)
]
fp16_linear_nodes = [
    node for node in linear_onnx_nodes if not is_qdq_wrapped_weighted_op(node)
]
def normalized_onnx_node_name(node):
    return node.name.replace("/", ".")

missing_int8_linear_module_names = [
    module_name for module_name in expected_int8_linear_names
    if not any(module_name in normalized_onnx_node_name(node) for node in wrapped_linear_nodes)
]
unexpected_int8_linear_nodes = [
    node.name for node in wrapped_linear_nodes
    if not any(module_name in normalized_onnx_node_name(node)
               for module_name in expected_int8_linear_names)
]
wrapped_conv = len(wrapped_conv_nodes)
if (
    not q_nodes
    or wrapped_conv != int8_conv_count
    or len(fp16_onnx_conv_nodes) != len(fp16_fallback_conv_names)
    or unexpected_fp16_nodes
    or unexpected_int8_nodes
    or len(wrapped_linear_nodes) != int8_linear_count
    or missing_int8_linear_module_names
    or unexpected_int8_linear_nodes
):
    raise RuntimeError({
        "conv": len(conv_nodes),
        "conv_qdq_wrapped": wrapped_conv,
        "fp16_conv": [node.name for node in fp16_onnx_conv_nodes],
        "unexpected_fp16_nodes": unexpected_fp16_nodes,
        "unexpected_int8_nodes": unexpected_int8_nodes,
        "matmul_gemm": len(linear_onnx_nodes),
        "matmul_gemm_qdq_wrapped": [node.name for node in wrapped_linear_nodes],
        "missing_int8_linear_module_names": missing_int8_linear_module_names,
        "unexpected_int8_linear_nodes": unexpected_int8_linear_nodes,
        "quantize_linear": len(q_nodes),
        "dequantize_linear": len(dq_nodes),
    })

onnx_audit = {
    "conv": len(conv_nodes),
    "conv_qdq_wrapped_int8": wrapped_conv,
    "conv_fp16_fallback": len(fp16_onnx_conv_nodes),
    "fp16_fallback_conv_names": [node.name for node in fp16_onnx_conv_nodes],
    "matmul_gemm_total": len(linear_onnx_nodes),
    "matmul_gemm_qdq_wrapped_int8": len(wrapped_linear_nodes),
    "matmul_gemm_qdq_wrapped_names": [node.name for node in wrapped_linear_nodes],
    "matmul_gemm_fp16": len(fp16_linear_nodes),
    "int8_linear_module_names": expected_int8_linear_names,
    "missing_int8_linear_module_names": missing_int8_linear_module_names,
    "unexpected_int8_linear_nodes": unexpected_int8_linear_nodes,
    "quantize_linear": len(q_nodes),
    "dequantize_linear": len(dq_nodes),
    "constant_weight_qdq_fold": weight_fold,
    "onnx_checker": "passed",
}

# Prima di TensorRT, l'ONNX deve riprodurre il fake-quant PyTorch su CUDA EP.
cuda_session = ort.InferenceSession(
    ONNX_PATH, providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)
sample_np = sample.detach().cpu().numpy().astype(np.float32, copy=False)
cuda_output = cuda_session.run(["logits"], {"images": sample_np})[0]
export_parity = {
    "max_abs_error": float(np.max(np.abs(cuda_output - fp16_fallback_output))),
    "mean_abs_error": float(np.mean(np.abs(cuda_output - fp16_fallback_output))),
    "prediction_agreement": float(np.mean(
        cuda_output.argmax(1) == fp16_fallback_output.argmax(1)
    )),
}
if export_parity["prediction_agreement"] < EXPORT_MIN_PREDICTION_AGREEMENT:
    raise RuntimeError(f"Export ONNX non fedele al fake-quant: {export_parity}")

RESULTS["artifacts"]["onnx_qat_int8"] = {
    "path": ONNX_PATH,
    "size_mib": Path(ONNX_PATH).stat().st_size / 2**20,
    "opset": ONNX_OPSET,
    "audit": onnx_audit,
    "fp16_fallback_parity_with_fp32_fake_quant": fallback_parity,
    "parity_with_pytorch_fp16_fallback": export_parity,
}

if RUN_TRT_FP16_BASELINE:
    # Usa esattamente i pesi post-QAT, ma disabilita ogni fake quantizer: cambia solo la precisione.
    fp16_baseline_model = copy.deepcopy(student).eval()
    disabled_quantizers = 0
    for module in fp16_baseline_model.modules():
        if isinstance(module, TensorQuantizer):
            module.disable()
            disabled_quantizers += 1
    still_enabled = [
        name for name, module in fp16_baseline_model.named_modules()
        if isinstance(module, TensorQuantizer) and module.is_enabled
    ]
    if still_enabled:
        raise RuntimeError({"fp16_baseline_quantizers_still_enabled": still_enabled})

    fp16_baseline_export_model = FP32IOFP16Fallback(fp16_baseline_model.half()).eval()
    with torch.inference_mode():
        fp16_baseline_output = fp16_baseline_export_model(sample).detach().cpu().numpy()
        torch.onnx.export(
            fp16_baseline_export_model,
            sample,
            FP16_ONNX_PATH,
            input_names=["images"],
            output_names=["logits"],
            opset_version=ONNX_OPSET,
            dynamo=False,
            do_constant_folding=True,
        )

    fp16_onnx_model = onnx.load(FP16_ONNX_PATH)
    onnx.checker.check_model(fp16_onnx_model, full_check=True)
    fp16_qdq_nodes = [
        node.name for node in fp16_onnx_model.graph.node
        if node.op_type in {"QuantizeLinear", "DequantizeLinear"}
    ]
    fp16_conv_count = sum(node.op_type == "Conv" for node in fp16_onnx_model.graph.node)
    if fp16_qdq_nodes or fp16_conv_count != conv_count:
        raise RuntimeError({
            "fp16_qdq_nodes": fp16_qdq_nodes[:10],
            "expected_conv": conv_count,
            "fp16_conv": fp16_conv_count,
        })

    fp16_cuda_session = ort.InferenceSession(
        FP16_ONNX_PATH, providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
    )
    fp16_cuda_output = fp16_cuda_session.run(["logits"], {"images": sample_np})[0]
    fp16_export_parity = {
        "max_abs_error": float(np.max(np.abs(fp16_cuda_output - fp16_baseline_output))),
        "mean_abs_error": float(np.mean(np.abs(fp16_cuda_output - fp16_baseline_output))),
        "prediction_agreement": float(np.mean(
            fp16_cuda_output.argmax(1) == fp16_baseline_output.argmax(1)
        )),
    }
    if fp16_export_parity["prediction_agreement"] < EXPORT_MIN_PREDICTION_AGREEMENT:
        raise RuntimeError(f"Export ONNX FP16 non fedele: {fp16_export_parity}")

    RESULTS["artifacts"]["onnx_qat_weights_fp16"] = {
        "path": FP16_ONNX_PATH,
        "size_mib": Path(FP16_ONNX_PATH).stat().st_size / 2**20,
        "opset": ONNX_OPSET,
        "source_weights": "same_best_qat_checkpoint_as_int8",
        "disabled_tensor_quantizers": disabled_quantizers,
        "quantize_dequantize_nodes": 0,
        "conv": fp16_conv_count,
        "onnx_checker": "passed",
        "parity_with_pytorch_fp16": fp16_export_parity,
    }
save_results()
print(json.dumps(RESULTS["artifacts"], indent=2))

## 11. TensorRT INT8 Q/DQ e confronto FP16 opzionale

Gli engine vengono costruiti direttamente con TensorRT e una cache identificata da hash ONNX, modalità e versione runtime. Accuratezza, latenza batch-1, energia NVML e memoria sono misurate sul forward completo. La baseline FP16 opzionale usa gli stessi pesi post-QAT con tutti i quantizer disabilitati.

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def build_tensorrt_engine(onnx_path, engine_path, build_mode):
    # La modalità entra nella chiave cache per non riusare un engine INT8 come baseline FP16.
    model_hash = sha256(onnx_path)
    engine_path = Path(engine_path)
    metadata_path = engine_path.with_suffix(engine_path.suffix + ".json")
    cache_hit = False
    if engine_path.is_file() and metadata_path.is_file():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        cache_hit = (
            metadata.get("onnx_sha256") == model_hash
            and metadata.get("build_mode") == build_mode
            and metadata.get("tensorrt_version") == trt.__version__
            and metadata.get("builder_optimization_level")
            == TRT_BUILDER_OPTIMIZATION_LEVEL
        )

    if not cache_hit:
        logger = trt.Logger(trt.Logger.WARNING)
        builder = trt.Builder(logger)
        if build_mode == "strongly_typed":
            network_flags = 1 << int(trt.NetworkDefinitionCreationFlag.STRONGLY_TYPED)
        elif build_mode in {"weakly_typed_explicit_qdq", "weakly_typed_fp16"}:
            # Il grafo resta explicit-Q/DQ; il weak typing amplia le tattiche del builder.
            network_flags = 0
        else:
            raise ValueError(f"Modalità TensorRT non supportata: {build_mode}")
        network = builder.create_network(network_flags)
        parser = trt.OnnxParser(network, logger)
        if not parser.parse_from_file(str(onnx_path)):
            errors = [str(parser.get_error(index)) for index in range(parser.num_errors)]
            raise RuntimeError({"stage": "onnx_parse", "errors": errors})

        config = builder.create_builder_config()
        if build_mode in {"weakly_typed_explicit_qdq", "weakly_typed_fp16"}:
            # Nel weak typing il grafo impone alcuni tensori Half: abilita le tattiche FP16.
            # INT8 non va abilitato qui: la quantizzazione è già espressa dai nodi Q/DQ.
            config.set_flag(trt.BuilderFlag.FP16)
        config.builder_optimization_level = TRT_BUILDER_OPTIMIZATION_LEVEL
        config.set_memory_pool_limit(
            trt.MemoryPoolType.WORKSPACE, TRT_WORKSPACE_GIB * 2**30
        )
        serialized_engine = builder.build_serialized_network(network, config)
        if serialized_engine is None:
            raise RuntimeError(
                f"TensorRT non ha prodotto l'engine in modalità {build_mode}"
            )
        engine_path.write_bytes(bytes(serialized_engine))
        metadata_path.write_text(
            json.dumps(
                {
                    "onnx_sha256": model_hash,
                    "build_mode": build_mode,
                    "tensorrt_version": trt.__version__,
                    "builder_optimization_level": TRT_BUILDER_OPTIMIZATION_LEVEL,
                },
                indent=2,
            ),
            encoding="utf-8",
        )

    return {
        "path": str(engine_path),
        "size_mib": engine_path.stat().st_size / 2**20,
        "onnx_sha256": model_hash,
        "cache_hit": cache_hit,
        "build_mode": build_mode,
        "strongly_typed": build_mode == "strongly_typed",
        "tensorrt_version": trt.__version__,
        "fp16_builder_flag": build_mode in {"weakly_typed_explicit_qdq", "weakly_typed_fp16"},
        "builder_optimization_level": TRT_BUILDER_OPTIMIZATION_LEVEL,
    }


TRT_TO_TORCH_DTYPE = {
    trt.float32: torch.float32,
    trt.float16: torch.float16,
    trt.int8: torch.int8,
    trt.int32: torch.int32,
    trt.bool: torch.bool,
}


class TensorRTRunner:
    def __init__(self, engine_path):
        self.logger = trt.Logger(trt.Logger.WARNING)
        self.runtime = trt.Runtime(self.logger)
        self.engine = self.runtime.deserialize_cuda_engine(Path(engine_path).read_bytes())
        if self.engine is None:
            raise RuntimeError("Impossibile deserializzare l'engine TensorRT")
        self.context = self.engine.create_execution_context()
        self.stream = torch.cuda.Stream(device=TRT_DEVICE_ID)
        names = [self.engine.get_tensor_name(index) for index in range(self.engine.num_io_tensors)]
        inputs = [name for name in names if self.engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT]
        outputs = [name for name in names if self.engine.get_tensor_mode(name) == trt.TensorIOMode.OUTPUT]
        if inputs != ["images"] or outputs != ["logits"]:
            raise RuntimeError({"engine_inputs": inputs, "engine_outputs": outputs})
        self.input_name, self.output_name = inputs[0], outputs[0]
        self.input_dtype = TRT_TO_TORCH_DTYPE[self.engine.get_tensor_dtype(self.input_name)]
        self.output_dtype = TRT_TO_TORCH_DTYPE[self.engine.get_tensor_dtype(self.output_name)]
        self.output = None

    @torch.inference_mode()
    def run(self, images):
        with torch.cuda.stream(self.stream):
            inputs = images.to(
                device=f"cuda:{TRT_DEVICE_ID}", dtype=self.input_dtype, non_blocking=True
            ).contiguous()
            engine_input_shape = tuple(self.engine.get_tensor_shape(self.input_name))
            if -1 in engine_input_shape:
                if not self.context.set_input_shape(self.input_name, tuple(inputs.shape)):
                    raise RuntimeError(f"Shape TensorRT non valida: {tuple(inputs.shape)}")
            elif tuple(inputs.shape) != engine_input_shape:
                raise ValueError({"expected": engine_input_shape, "received": tuple(inputs.shape)})

            output_shape = tuple(self.context.get_tensor_shape(self.output_name))
            if any(dimension < 0 for dimension in output_shape):
                raise RuntimeError(f"Shape output TensorRT irrisolta: {output_shape}")
            if self.output is None or tuple(self.output.shape) != output_shape:
                self.output = torch.empty(
                    output_shape, device=f"cuda:{TRT_DEVICE_ID}", dtype=self.output_dtype
                )

            self.context.set_tensor_address(self.input_name, inputs.data_ptr())
            self.context.set_tensor_address(self.output_name, self.output.data_ptr())
            if not self.context.execute_async_v3(stream_handle=self.stream.cuda_stream):
                raise RuntimeError("Esecuzione TensorRT fallita")
        self.stream.synchronize()
        return self.output


@torch.inference_mode()
def evaluate_tensorrt(runner, loader, split):
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.int64)
    for batch in tqdm(loader, desc=f"TensorRT {split}"):
        update_confusion(confusion, runner.run(batch["image"]).argmax(1), batch["mask"])
    metrics = segmentation_metrics(confusion)
    metrics.update(split=split, samples=len(loader.dataset))
    return metrics


def measure_latency(runner, images):
    if images.shape[0] != 1:
        raise ValueError("Le misure di deployment richiedono batch size 1")
    images = images.to(device=f"cuda:{TRT_DEVICE_ID}", dtype=runner.input_dtype)
    run_once = lambda: runner.run(images)
    for _ in range(LATENCY_WARMUP_RUNS):
        run_once()
    values = []
    for _ in range(LATENCY_MEASURED_RUNS):
        start = time.perf_counter()
        run_once()
        values.append((time.perf_counter() - start) * 1000.0)
    mean_ms = float(np.mean(values))
    return {
        "mean_ms": mean_ms,
        "median_ms": float(np.median(values)),
        "p95_ms": float(np.percentile(values, 95)),
        "images_per_second": 1000.0 / mean_ms,
    }


class NvmlMonitor:
    def __init__(self):
        self.stop_event = threading.Event()
        self.samples = []
        self.error = None

    def _memory_mib(self):
        memory = pynvml.nvmlDeviceGetMemoryInfo(self.handle)
        try:
            for process in pynvml.nvmlDeviceGetComputeRunningProcesses(self.handle):
                used = process.usedGpuMemory
                if process.pid == os.getpid() and isinstance(used, (int, np.integer)):
                    if 0 <= int(used) <= int(memory.total):
                        return int(used) / 2**20
        except (AttributeError, pynvml.NVMLError):
            pass
        return memory.used / 2**20

    def _sample(self):
        self.samples.append((
            time.perf_counter(),
            pynvml.nvmlDeviceGetPowerUsage(self.handle) / 1000.0,
            self._memory_mib(),
        ))

    def _loop(self):
        try:
            while not self.stop_event.wait(NVML_SAMPLE_INTERVAL_SECONDS):
                self._sample()
        except Exception as error:
            self.error = error
            self.stop_event.set()

    def start(self):
        pynvml.nvmlInit()
        self.handle = pynvml.nvmlDeviceGetHandleByIndex(TRT_DEVICE_ID)
        try:
            self.energy_start_mj = pynvml.nvmlDeviceGetTotalEnergyConsumption(self.handle)
        except (AttributeError, pynvml.NVMLError):
            self.energy_start_mj = None
        self._sample()
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def stop(self):
        try:
            energy_end_mj = (
                pynvml.nvmlDeviceGetTotalEnergyConsumption(self.handle)
                if self.energy_start_mj is not None else None
            )
        except (AttributeError, pynvml.NVMLError):
            energy_end_mj = None
        self.stop_event.set()
        self.thread.join(timeout=2.0)
        self._sample()
        if self.error:
            raise RuntimeError(self.error)
        if energy_end_mj is not None:
            energy_j = (energy_end_mj - self.energy_start_mj) / 1000.0
        else:
            energy_j = sum(
                0.5 * (left[1] + right[1]) * (right[0] - left[0])
                for left, right in zip(self.samples, self.samples[1:])
            )
        return energy_j


def measure_efficiency(runner, images):
    if images.shape[0] != 1:
        raise ValueError("Le misure di deployment richiedono batch size 1")
    images = images.to(device=f"cuda:{TRT_DEVICE_ID}", dtype=runner.input_dtype)
    run_once = lambda: runner.run(images)
    for _ in range(EFFICIENCY_WARMUP_RUNS):
        run_once()
    monitor = NvmlMonitor()
    monitor.start()
    runs = 0
    start = time.perf_counter()
    try:
        while runs < EFFICIENCY_MIN_RUNS or time.perf_counter() - start < EFFICIENCY_MIN_SECONDS:
            run_once()
            runs += 1
    finally:
        energy_j = monitor.stop()
    energy_per_image_mj = energy_j * 1000.0 / runs
    return {
        "average_sampled_power_w": float(np.mean([sample[1] for sample in monitor.samples])),
        "energy_per_image_mj": float(energy_per_image_mj),
        "images_per_joule": float(1000.0 / energy_per_image_mj),
        "peak_gpu_memory_mib": float(max(sample[2] for sample in monitor.samples)),
    }


# Entrambi gli engine sono TensorRT diretti e usano gli stessi pesi post-QAT.
engine_info = build_tensorrt_engine(ONNX_PATH, ENGINE_PATH, TRT_BUILD_MODE)
fp16_engine_info = (
    build_tensorrt_engine(FP16_ONNX_PATH, FP16_ENGINE_PATH, TRT_FP16_BUILD_MODE)
    if RUN_TRT_FP16_BASELINE else None
)

int8_probe_runner = TensorRTRunner(ENGINE_PATH)
trt_output = int8_probe_runner.run(sample).float().cpu().numpy()
trt_parity = {
    "max_abs_error": float(np.max(np.abs(trt_output - fp16_fallback_output))),
    "mean_abs_error": float(np.mean(np.abs(trt_output - fp16_fallback_output))),
    "prediction_agreement": float(np.mean(
        trt_output.argmax(1) == fp16_fallback_output.argmax(1)
    )),
}
if trt_parity["prediction_agreement"] < TRT_MIN_PREDICTION_AGREEMENT:
    raise RuntimeError(f"TensorRT non fedele al fake-quant QAT: {trt_parity}")
del int8_probe_runner

if RUN_TRT_FP16_BASELINE:
    fp16_probe_runner = TensorRTRunner(FP16_ENGINE_PATH)
    fp16_trt_output = fp16_probe_runner.run(sample).float().cpu().numpy()
    fp16_trt_parity = {
        "max_abs_error": float(np.max(np.abs(fp16_trt_output - fp16_baseline_output))),
        "mean_abs_error": float(np.mean(np.abs(fp16_trt_output - fp16_baseline_output))),
        "prediction_agreement": float(np.mean(
            fp16_trt_output.argmax(1) == fp16_baseline_output.argmax(1)
        )),
    }
    if fp16_trt_parity["prediction_agreement"] < TRT_MIN_PREDICTION_AGREEMENT:
        raise RuntimeError(f"TensorRT FP16 non fedele al PyTorch FP16: {fp16_trt_parity}")
    del fp16_probe_runner

# I modelli PyTorch escono dalla GPU prima delle misure NVML per non sporcare la memoria di picco.
teacher.cpu()
student.cpu()
del cuda_session, export_model, sample
if RUN_TRT_FP16_BASELINE:
    del fp16_cuda_session, fp16_baseline_export_model, fp16_baseline_model
gc.collect()
torch.cuda.empty_cache()

if RUN_TRT_FP16_BASELINE:
    # La baseline viene misurata e poi scaricata dalla GPU prima di creare il runner INT8.
    fp16_runner = TensorRTRunner(FP16_ENGINE_PATH)
    fp16_benchmark = {
        "runtime": "tensorrt",
        "execution": fp16_engine_info["build_mode"],
        "precision": "fp16_same_qat_trained_weights",
        "mixed_provider_execution": False,
        "parity_with_pytorch_fp16": fp16_trt_parity,
        "metrics": evaluate_tensorrt(fp16_runner, val_loader, VAL_SPLIT),
        "latency": measure_latency(fp16_runner, next(iter(val_loader))["image"]),
        "efficiency": measure_efficiency(fp16_runner, next(iter(val_loader))["image"]),
        "engine": fp16_engine_info,
    }
    RESULTS["artifacts"]["tensorrt_engine_fp16"] = fp16_engine_info
    RESULTS["benchmarks"]["onnx_modelopt_qat_weights_fp16_tensorrt"] = fp16_benchmark
    del fp16_runner
    gc.collect()
    torch.cuda.empty_cache()

trt_runner = TensorRTRunner(ENGINE_PATH)
int8_benchmark = {
    "runtime": "tensorrt",
    "execution": engine_info["build_mode"],
    "precision": "int8_qdq_with_fp16_fallback",
    "mixed_provider_execution": False,
    "parity_with_pytorch_fp16_fallback": trt_parity,
    "metrics": evaluate_tensorrt(trt_runner, val_loader, VAL_SPLIT),
    "latency": measure_latency(trt_runner, next(iter(val_loader))["image"]),
    "efficiency": measure_efficiency(trt_runner, next(iter(val_loader))["image"]),
    "engine": engine_info,
}
RESULTS["artifacts"]["tensorrt_engine_int8"] = engine_info
RESULTS["benchmarks"]["onnx_modelopt_qat_int8_tensorrt"] = int8_benchmark

def reduction_percent(current_value, reference_value):
    return float(100.0 * (reference_value - current_value) / reference_value)


def comparison_snapshot(benchmark):
    return {
        "validation_miou": benchmark["metrics"]["miou"],
        "mean_latency_ms": benchmark["latency"]["mean_ms"],
        "p95_latency_ms": benchmark["latency"]["p95_ms"],
        "average_sampled_power_w": benchmark["efficiency"]["average_sampled_power_w"],
        "energy_per_image_mj": benchmark["efficiency"]["energy_per_image_mj"],
        "images_per_joule": benchmark["efficiency"]["images_per_joule"],
        "peak_gpu_memory_mib": benchmark["efficiency"]["peak_gpu_memory_mib"],
        "engine_size_mib": benchmark["engine"]["size_mib"],
    }


current_snapshot = comparison_snapshot(int8_benchmark)
conv_only_reference = dict(SEGFORMER_QAT_CONV_ONLY_REFERENCE)
vs_conv_only = {
    "protocol": "confronto con il QAT SegFormer Conv-only già salvato, stesso split e protocollo",
    "baseline_variant": "int8_conv_only_qdq",
    "current_variant": QAT_VARIANT,
    "baseline": conv_only_reference,
    "current": current_snapshot,
    "validation_miou_delta_current_minus_conv_only": float(
        current_snapshot["validation_miou"] - conv_only_reference["validation_miou"]
    ),
    "mean_latency_reduction_percent": reduction_percent(
        current_snapshot["mean_latency_ms"], conv_only_reference["mean_latency_ms"]
    ),
    "p95_latency_reduction_percent": reduction_percent(
        current_snapshot["p95_latency_ms"], conv_only_reference["p95_latency_ms"]
    ),
    "average_power_reduction_percent": reduction_percent(
        current_snapshot["average_sampled_power_w"],
        conv_only_reference["average_sampled_power_w"],
    ),
    "energy_per_image_reduction_percent": reduction_percent(
        current_snapshot["energy_per_image_mj"],
        conv_only_reference["energy_per_image_mj"],
    ),
    "images_per_joule_increase_percent": float(100.0 * (
        current_snapshot["images_per_joule"]
        / conv_only_reference["images_per_joule"] - 1.0
    )),
    "peak_gpu_memory_reduction_percent": reduction_percent(
        current_snapshot["peak_gpu_memory_mib"],
        conv_only_reference["peak_gpu_memory_mib"],
    ),
    "engine_size_reduction_percent": reduction_percent(
        current_snapshot["engine_size_mib"], conv_only_reference["engine_size_mib"]
    ),
}
vs_conv_only["validation_accuracy_guardrail"] = bool(
    current_snapshot["validation_miou"]
    >= conv_only_reference["validation_miou"] - MAX_VALIDATION_MIOU_DROP_VS_CONV_ONLY
)
vs_conv_only["energy_improved"] = bool(
    current_snapshot["energy_per_image_mj"] < conv_only_reference["energy_per_image_mj"]
)
vs_conv_only["selected_on_validation"] = bool(
    vs_conv_only["validation_accuracy_guardrail"] and vs_conv_only["energy_improved"]
)
RESULTS["comparisons"]["vs_segformer_qat_conv_only"] = vs_conv_only

if RUN_TRT_FP16_BASELINE:
    int8_vs_fp16 = {
        "protocol": "stessi pesi post-QAT, TensorRT diretto, batch 1 e stesso input",
        "measurement_order": ["fp16", "int8"],
        "fp16": comparison_snapshot(fp16_benchmark),
        "int8": comparison_snapshot(int8_benchmark),
        "validation_miou_delta_int8_minus_fp16": float(
            int8_benchmark["metrics"]["miou"] - fp16_benchmark["metrics"]["miou"]
        ),
        "mean_latency_reduction_percent": reduction_percent(
            int8_benchmark["latency"]["mean_ms"], fp16_benchmark["latency"]["mean_ms"]
        ),
        "p95_latency_reduction_percent": reduction_percent(
            int8_benchmark["latency"]["p95_ms"], fp16_benchmark["latency"]["p95_ms"]
        ),
        "average_power_reduction_percent": reduction_percent(
            int8_benchmark["efficiency"]["average_sampled_power_w"],
            fp16_benchmark["efficiency"]["average_sampled_power_w"],
        ),
        "energy_per_image_reduction_percent": reduction_percent(
            int8_benchmark["efficiency"]["energy_per_image_mj"],
            fp16_benchmark["efficiency"]["energy_per_image_mj"],
        ),
        "images_per_joule_increase_percent": float(100.0 * (
            int8_benchmark["efficiency"]["images_per_joule"]
            / fp16_benchmark["efficiency"]["images_per_joule"] - 1.0
        )),
        "peak_gpu_memory_reduction_percent": reduction_percent(
            int8_benchmark["efficiency"]["peak_gpu_memory_mib"],
            fp16_benchmark["efficiency"]["peak_gpu_memory_mib"],
        ),
        "engine_size_reduction_percent": reduction_percent(
            engine_info["size_mib"], fp16_engine_info["size_mib"]
        ),
    }
    RESULTS["comparisons"]["int8_vs_fp16_tensorrt"] = int8_vs_fp16
save_results()
print(json.dumps({
    "fp16": RESULTS["benchmarks"].get("onnx_modelopt_qat_weights_fp16_tensorrt"),
    "int8": int8_benchmark,
    "vs_qat_conv_only": RESULTS["comparisons"]["vs_segformer_qat_conv_only"],
    "vs_fp16": RESULTS["comparisons"].get("int8_vs_fp16_tensorrt"),
}, indent=2))

## 12. Test hold-out finale

Il test viene caricato ed eseguito soltanto quando `RUN_FINAL_TEST=True`, dopo la selezione sulla validation. La cella confronta teacher FP32, student fake-quant e TensorRT sullo stesso split, senza riutilizzare il test per modificare la configurazione.

In [ ]:
if RUN_FINAL_TEST:
    test_dataset = S5MarsDataset(TEST_SPLIT, MAX_TEST_SAMPLES)
    test_loader = make_loader(test_dataset, EVAL_BATCH_SIZE, shuffle=False)
    final_test = {
        "protocol": "hold-out finale; non usare per scegliere configurazione QAT",
        "split": TEST_SPLIT,
        "samples": len(test_dataset),
    }

    teacher.to(DEVICE).eval()
    final_test["pytorch_fp32_cuda"] = evaluate_torch(teacher, test_loader, TEST_SPLIT)
    teacher.cpu()
    student.to(DEVICE).eval()
    final_test["pytorch_modelopt_qat_fake_quant"] = evaluate_torch(student, test_loader, TEST_SPLIT)
    student.cpu()
    gc.collect()
    torch.cuda.empty_cache()
    final_test["onnx_modelopt_qat_int8_tensorrt"] = evaluate_tensorrt(
        trt_runner, test_loader, TEST_SPLIT
    )
    RESULTS["comparisons"]["vs_segformer_qat_conv_only"][
        "test_miou_delta_current_minus_conv_only"
    ] = float(
        final_test["onnx_modelopt_qat_int8_tensorrt"]["miou"]
        - SEGFORMER_QAT_CONV_ONLY_REFERENCE["test_miou"]
    )
    if RUN_TRT_FP16_BASELINE:
        fp16_test_runner = TensorRTRunner(FP16_ENGINE_PATH)
        final_test["onnx_modelopt_qat_weights_fp16_tensorrt"] = evaluate_tensorrt(
            fp16_test_runner, test_loader, TEST_SPLIT
        )
        del fp16_test_runner
        RESULTS["comparisons"]["int8_vs_fp16_tensorrt"][
            "test_miou_delta_int8_minus_fp16"
        ] = float(
            final_test["onnx_modelopt_qat_int8_tensorrt"]["miou"]
            - final_test["onnx_modelopt_qat_weights_fp16_tensorrt"]["miou"]
        )
    RESULTS["test_metrics"] = final_test
    save_results()
    print(json.dumps(final_test, indent=2))
else:
    print("Test non eseguito: impostare RUN_FINAL_TEST=True soltanto dopo la selezione sulla validation.")

## 13. Salvataggio finale

Riepiloga i benchmark già raccolti, salva `results.json` e carica gli artefatti richiesti su Drive. Ogni file mantiene un nome specifico della variante per evitare sovrascritture tra scope INT8 differenti.

In [ ]:
def print_summary(results):
    for name, item in results["benchmarks"].items():
        metrics = item.get("metrics", {})
        latency = item.get("latency", {})
        efficiency = item.get("efficiency", {})
        print(
            name,
            "mIoU=", metrics.get("miou"),
            "mean_ms=", latency.get("mean_ms"),
            "mJ/img=", efficiency.get("energy_per_image_mj"),
            "peak_MiB=", efficiency.get("peak_gpu_memory_mib"),
        )
    comparison = results.get("training", {}).get("pre_post_qat_validation")
    if comparison is not None:
        print("training.pre_post_qat_validation")
        print(json.dumps(comparison, indent=2))
    for name, value in results.get("comparisons", {}).items():
        print(f"comparisons.{name}")
        print(json.dumps(value, indent=2))


save_results()
print_summary(RESULTS)

if UPLOAD_FINAL_ARTIFACTS:
    service = drive_service()
    drive_folder_id = resolve_output_folder(service)
    # L'engine è legato a GPU e versione TensorRT: si rigenera dall'ONNX e non si archivia.
    final_paths = [BEST_QAT_PATH, LAST_QAT_PATH, HISTORY_PATH, ONNX_PATH, RESULTS_PATH]
    if RUN_TRT_FP16_BASELINE:
        final_paths.append(FP16_ONNX_PATH)
    for path in final_paths:
        if RESTORE_SAVED_QAT and not Path(path).is_file():
            continue
        upload_file(path, service, drive_folder_id)
    print("Artefatti salvati in Drive:", QAT_DRIVE_FOLDER_NAME)